In [1]:
import os
import inspect
import seaborn
import matplotlib
import matplotlib.pyplot as plt
import torch
import scanpy as sc
import pyro

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("GPU is available")
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available")

import antipode
from antipode.antipode_model import *
import antipode.model_functions
from antipode.model_functions import *
import antipode.model_distributions
from antipode.model_distributions import *
import antipode.model_modules
from antipode.model_modules import *
import antipode.train_utils
from antipode.train_utils import *
import antipode.plotting
from antipode.plotting import *

/home/matthew.schmitz/Matthew/utils/miniforge3/envs/antipode/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU is not available


In [ ]:
adata=sc.read_h5ad(os.path.expanduser('/allen/programs/celltypes/workgroups/rnaseqanalysis/EvoGen/Team/Matthew/data/cortex_data/jorstad_merge_all.h5ad'),backed='r')
adata.uns['species_colors']=['blue','red','green','yellow','orange','cyan','magenta','teal','goldenrod','purple','maroon','pink','violet','black','grey','beige']
batch_key='batch'
discov_key='organism'
layer_key='UMIs'


In [2]:
adata=sc.read_h5ad(os.path.expanduser('/allen/programs/celltypes/workgroups/rnaseqanalysis/EvoGen/Team/Matthew/data/cortex_data/jorstad_merge_all.h5ad'),backed='r')

/home/matthew.schmitz/Matthew/utils/miniforge3/envs/antipode/lib/python3.11/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [2]:
import anndata
jor_path = '/allen/programs/celltypes/workgroups/rnaseqanalysis/EvoGen/Team/Matthew/data/cortex_data/jorstad_cross_areal/'
jorstads = os.listdir(jor_path)
adatas = []
for f in jorstads:
    if '.h5ad' in f and not 'smart_seq' in f:
        adatas.append(sc.read_h5ad(os.path.join(jor_path,f)))

In [24]:
for i in range(len(adatas)):
    if not 'ensembl_id' in adatas[i].var.columns:
        print(adatas[i].var)
        adatas[i].var['ensembl_id']=list(adatas[i].var.index)
        adatas[i].var.index=list(adatas[i].var['feature_name'])

                 feature_is_filtered feature_name feature_reference  \
gene                                                                  
ENSG00000121410                False         A1BG    NCBITaxon:9606   
ENSG00000268895                False     A1BG-AS1    NCBITaxon:9606   
ENSG00000148584                False         A1CF    NCBITaxon:9606   
ENSG00000175899                False          A2M    NCBITaxon:9606   
ENSG00000245105                False      A2M-AS1    NCBITaxon:9606   
...                              ...          ...               ...   
ENSG00000204479                False     PRAMEF17    NCBITaxon:9606   
ENSG00000280267                False     PRAMEF26    NCBITaxon:9606   
ENSG00000188655                False       RNASE9    NCBITaxon:9606   
ENSG00000275954                False      TBC1D3F    NCBITaxon:9606   
ENSG00000228567                False        VN1R4    NCBITaxon:9606   

                feature_biotype  
gene                             
ENSG0000

In [25]:
adata=anndata.concat(adatas,merge='same')

/home/matthew.schmitz/Matthew/utils/miniforge3/envs/antipode/lib/python3.11/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [26]:
adata.write('/allen/programs/celltypes/workgroups/rnaseqanalysis/EvoGen/Team/Matthew/data/cortex_data/jorstad_merge_xa.h5ad')

In [ ]:
num_var=adata.shape[1]
batch_size=32
level_sizes=[1,25,200]
num_latent=200
steps=0
max_steps=300000
print_every=5000
model_tag='1.9.1.8.3_JorstadAll-SCVI'

In [ ]:
scvi.model.SCVI.setup_anndata(adata, layer=layer_key, batch_key=batch_key,categorical_covariate_keys=[discov_key])#'roi',,
scvi_model = scvi.model.SCVI (adata, n_latent=num_latent,n_layers=3,n_hidden=2048)
scvi_model.train(100)
SCVI_LATENT_KEY = "X_SCVI"
adata.obsm[SCVI_LATENT_KEY] = scvi_model.get_latent_representation(adata)
scvi_model.save(os.path.join('/allen/programs/celltypes/workgroups/rnaseqanalysis/EvoGen/Team/Matthew/models/',model_tag),overwrite=True) 

In [ ]:
MDE_KEY = "X_SCVI_MDE"
adata.obsm[MDE_KEY] = clip_latent_dimensions(scvi.model.utils.mde(adata.obsm['X_SCVI'],init='random'),0.1)
sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=["species"],legend_fontsize=6,legend_fontweight='normal',
    legend_loc='on data',palette=sc.pl.palettes.godsnot_102
)

In [ ]:

# Clear Pyro param store so we don't conflict with previous run
try:
    pyro.clear_param_store()
    del antipode_model
    torch.cuda.empty_cache()
except:
    pass
# Fix random number seed to a lucky number
pyro.util.set_rng_seed(13)
# Enable optional validation warnings
pyro.enable_validation(False)

# Instantiate instance of model/guide and various neural networks
antipode_model = ANTIPODE(num_latent=num_latent,level_sizes=level_sizes,bi_depth=2,num_batch_embed=10,psi_levels=[0.,1.,1.],
                adata=adata,discov_pair=('obs',discov_key),batch_pair=('obs',batch_key),layer=layer_key,
                use_psi=False,use_q_score=True,prior_scale=50.,sampler_category='organism',dist_normalize=False,
                scale_init_val=0.01,loc_as_param=False,zdw_as_param=False,intercept_as_param=False)


# Training Phase 2: Intializing layered tree

In [ ]:
antipode_model.prepare_phase_2(epochs=2,device=device,dimension_reduction='X_SCVI')

In [ ]:
antipode_model.train_phase(phase=2,max_steps=int(max_steps/2),print_every=10000,num_particles=3,device=device, max_learning_rate=5e-4, one_cycle_lr=True, batch_size=64,freeze_encoder=False)

In [ ]:
!nvidia-smi
antipode_model.to('cpu')
torch.cuda.empty_cache()
!nvidia-smi

In [ ]:
antipode_model.store_outputs(device=device,prefix='')

In [ ]:
!nvidia-smi
antipode_model.to('cpu')
torch.cuda.empty_cache()
!nvidia-smi

In [ ]:
plot_loss(antipode_model.losses)
plot_gmm_heatmaps(antipode_model)
plot_d_hists(antipode_model)
plot_tree_edge_weights(antipode_model)

In [ ]:
antipode_model.save(os.path.join('/home/matthew.schmitz/Matthew/models/',model_tag),save_anndata=False,prefix='p2_')

# Training Phase 3: Refining the final tree

In [ ]:
antipode_model.train_phase(phase=3,max_steps=max_steps,print_every=10000,num_particles=5,device=device, max_learning_rate=1e-3, one_cycle_lr=True, steps=0, batch_size=32)

In [ ]:
!nvidia-smi
antipode_model.to('cpu')
torch.cuda.empty_cache()
!nvidia-smi

In [ ]:
plot_loss(antipode_model.losses)

In [ ]:
antipode_model.store_outputs(device=device,prefix='')

In [ ]:
!nvidia-smi
antipode_model.to('cpu')
torch.cuda.empty_cache()
!nvidia-smi

In [ ]:
plot_gmm_heatmaps(antipode_model)
plot_d_hists(antipode_model)
plot_tree_edge_weights(antipode_model)

In [ ]:
MDE_KEY = "X_antipode_MDE"
adata.obsm[MDE_KEY] = clip_latent_dimensions(scvi.model.utils.mde(adata.obsm['X_antipode'],init='random'),0.1)
sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=["antipode_cluster","kmeans"],legend_fontsize=6,legend_fontweight='normal',
    legend_loc='on data',palette=sc.pl.palettes.godsnot_102
)

sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=[x for x in adata.obs.columns if 'level' in x],
    palette=sc.pl.palettes.godsnot_102,
    legend_loc='on data'
)

sc.pl.embedding(
    adata,
    basis=MDE_KEY,
    color=["psi",'q_score',discov_key,batch_key],palette=sc.pl.palettes.godsnot_102,cmap='coolwarm'
)


In [ ]:
antipode_model.save(os.path.join('/home/matthew.schmitz/Matthew/models/',model_tag),save_anndata=True,prefix='p3_')

In [ ]:
seaborn.histplot(adata.obs['q_score'])

In [ ]:
random_choice=np.random.choice(adata.obs.index,size=100000,replace=False)
random_choice=np.where(adata.obs.index.isin(random_choice))[0]
xdata=adata[random_choice,:]
xdata=xdata.to_memory().copy()

In [ ]:
xdata.X=xdata.layers[layer_key]
sc.pp.normalize_per_cell(xdata)
sc.pp.log1p(xdata)
#sc.pp.scale(xdata,max_value=10)

gene_list=['RBFOX3','PDGFRA','AQP4','FOXJ1','AIF1','MOG','COL1A2','CD34','COL4A1','FOXG1','SATB2','RORB','SLC17A7','TLE4','FEZF2',
           'DLX2','PROX1','SCGN','NKX2-1','LHX6','SST','PVALB','CRABP1','MEIS2','TSHZ1','NPY','FOXP1','FOXP2','PDYN','PENK','ISL1',
           'MKI67','RPL7','RPS17','RPL13A','MEF2C',
           'HMX3','TH','LMX1A','TFAP2A','TFAP2B','RSPO1','NKX3-1','IGF1','ITPR2','OTX2','HOXB3','PAX1','PAX2','PAX3','PAX5','PAX6','PAX7','PAX8']
gene_list=[x for x in gene_list if x in xdata.var.index]
sc.pl.embedding(
    xdata,
    basis=MDE_KEY,
    color=gene_list,cmap='Purples',
    palette=sc.pl.palettes.godsnot_102,legend_fontsize=6,
    legend_loc='on data',use_raw=False
)
